# VN-GAT on Kaggle (2x T4)

**Settings -> Accelerator: `GPU T4 x2`**  
**Settings -> Internet: `On`** (needed for pip installs and Google Drive)

Two-stage reassembly: an SO(3)-equivariant GNN predicts per-fragment rotation, then a closed-form solver recovers translation.


### CELL 1: install the two packages Kaggle's image lacks


In [ ]:
# Everything else (torch, numpy, scipy, matplotlib) is already present.
!pip install -q libigl trimesh
# Only if you want checkpoints mirrored to Google Drive:
!pip install -q google-api-python-client google-auth


### CELL 2: get the code


In [ ]:
# Either upload vngat_project as a Kaggle Dataset and copy it out:
!cp -r /kaggle/input/vngat-code/vngat_project /kaggle/working/vngat
# ...or unzip an uploaded archive:
# !unzip -q /kaggle/input/vngat-code/vngat_project.zip -d /kaggle/working/
%cd /kaggle/working/vngat
!ls


### CELL 3: point at the data


In [ ]:
# Symlink whichever Breaking Bad dataset(s) you attached into ./data
!mkdir -p data
!ln -sfn /kaggle/input/breaking-bad-dataset-everyday/everyday_compressed data/everyday_compressed
!ln -sfn /kaggle/input/breaking-bad-dataset-artifact/artifact_compressed data/artifact_compressed
import sys; sys.path.insert(0, '/kaggle/working/vngat')
from vngat.data.splits import list_scene_directories
dirs = list_scene_directories("data")
print(f"{len(dirs)} scene directories found")
print(dirs[:3])


### CELL 4: sanity-check the pipeline before burning session time


In [ ]:
!python -m pytest tests -q -x


### CELL 5: measure throughput BEFORE planning a long run


In [ ]:
# Read the last line: data loading only bottlenecks once the per-batch time
# exceeds per-batch compute. Plan epochs from THIS number, not from a guess.
!python -m scripts.benchmark_data --root_dir data --num_scenes 25 --batch_size 2 --num_workers 2


### CELL 6 (optional): Google Drive, so checkpoints survive the session


In [ ]:
# Kaggle wipes /kaggle/working when the session ends.
#
#   1. Google Cloud console -> new project -> enable the Drive API.
#   2. Create a service account -> create a JSON key -> download it.
#   3. In your own Drive, make a folder for checkpoints, then SHARE it with the
#      service account's client_email as Editor. This step is mandatory: a
#      service account has no storage quota of its own, and skipping it fails
#      later with a confusing "storageQuotaExceeded".
#   4. Copy the folder id from its URL: drive.google.com/drive/folders/<ID>
#   5. Add-ons -> Secrets -> add a secret named GDRIVE_SA holding the JSON.
DRIVE_FOLDER_ID = ""      # <- paste the folder id
DRIVE_SECRET    = "GDRIVE_SA"

if DRIVE_FOLDER_ID:
    from vngat.training.drive import DriveSync
    probe = DriveSync(DRIVE_FOLDER_ID, DRIVE_SECRET)
    print("Drive reachable:", probe.enabled)


### CELL 7: train


In [ ]:
# Uses both T4s via DDP (one process per GPU). Stops cleanly at
# time_budget_hours so the session ends with a saved, resumable checkpoint
# rather than being killed mid-epoch.
#
# `resume: auto` means re-running this exact cell in a NEW session picks up
# where the last one stopped (pulling from Drive if configured).
args = [
    "--config", "configs/kaggle_2xt4_full.yaml",
    "--root_dir", "data",
    "--checkpoint_dir", "/kaggle/working/checkpoints",
    "--time_budget_hours", "11.0",
]
if DRIVE_FOLDER_ID:
    args += ["--drive_folder_id", DRIVE_FOLDER_ID, "--drive_credentials", DRIVE_SECRET]

import subprocess, sys
subprocess.run([sys.executable, "-m", "scripts.train", *args], check=True)

# For the fracture-surface variant, swap in configs/kaggle_2xt4_frac.yaml.


### CELL 8: curves


In [ ]:
!python -m scripts.plot_history --checkpoint_dir /kaggle/working/checkpoints --out curves.png
from IPython.display import Image; Image("curves.png")


### CELL 9: GARF-comparable evaluation


In [ ]:
# Reports Euler RMSE *and* geodesic angle separately -- they are different
# numbers and the thesis tables must say which one they quote.
!python -m scripts.evaluate \
    --checkpoint /kaggle/working/checkpoints/best.pt \
    --split test --num_scenes 200 \
    --out /kaggle/working/eval_test.json


### CELL 10: look at a prediction


In [ ]:
# left: scattered input | middle: model reassembly | right: ground truth
!python -m scripts.visualize --mode prediction \
    --checkpoint /kaggle/working/checkpoints/best.pt \
    --out /kaggle/working/prediction.glb
